In [13]:
import pandas as pd
from tools.sbd import sentencePipeline

from sentence_transformers import SentenceTransformer
import pickle
from tools.embedder import embed


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11791.08it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Data preprocessing

In [14]:
df = pd.read_csv("/home/prashanth/nlp/financial_assistant/dataset/IN-FINews Dataset.csv")

In [15]:
df.columns

Index(['Title', 'Date', 'Description', 'Author', 'Content', 'Keywords', 'URL'], dtype='str')

In [16]:
all_sentences = []
m = 'all-MiniLM-L6-v2'
for text in df['Description'].dropna():
    sentences = sentencePipeline(str(text))
    sentences = [x for x in sentences if x.strip() != ""]
    all_sentences.extend(sentences)

with open("dataset/sentences.txt", "w", encoding="utf-8") as f:
    for sentence in all_sentences:
        f.write(sentence.strip() + "\n")

In [17]:
print(f"No. of sentence before SBD {df.Description.shape[0]}")
print(f"No. of sentence after SBD {len(all_sentences)}")

No. of sentence before SBD 3348
No. of sentence after SBD 3408


# Embedding Generation

## 1. Using BERT

In [19]:
from sentence_transformers import SentenceTransformer
import torch

model = SentenceTransformer(m)
model.load_state_dict(torch.load('model/bert.pt', map_location='cpu'))
model.eval()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14126.39it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [26]:
model = SentenceTransformer('all-MiniLM-L6-v2')

input_file = "dataset/sentences.txt" 
output_file = "dataset/bert_embeddings.pkl"

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5061.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [27]:
embed(input_file, output_file)

found 3408 sentences
done 10 / 3408
done 20 / 3408
done 30 / 3408
done 40 / 3408
done 50 / 3408
done 60 / 3408
done 70 / 3408
done 80 / 3408
done 90 / 3408
done 100 / 3408
done 110 / 3408
done 120 / 3408
done 130 / 3408
done 140 / 3408
done 150 / 3408
done 160 / 3408
done 170 / 3408
done 180 / 3408
done 190 / 3408
done 200 / 3408
done 210 / 3408
done 220 / 3408
done 230 / 3408
done 240 / 3408
done 250 / 3408
done 260 / 3408
done 270 / 3408
done 280 / 3408
done 290 / 3408
done 300 / 3408
done 310 / 3408
done 320 / 3408
done 330 / 3408
done 340 / 3408
done 350 / 3408
done 360 / 3408
done 370 / 3408
done 380 / 3408
done 390 / 3408
done 400 / 3408
done 410 / 3408
done 420 / 3408
done 430 / 3408
done 440 / 3408
done 450 / 3408
done 460 / 3408
done 470 / 3408
done 480 / 3408
done 490 / 3408
done 500 / 3408
done 510 / 3408
done 520 / 3408
done 530 / 3408
done 540 / 3408
done 550 / 3408
done 560 / 3408
done 570 / 3408
done 580 / 3408
done 590 / 3408
done 600 / 3408
done 610 / 3408
done 620 / 3

## 2. Using FastText

In [1]:
import fasttext
import fasttext.util
import numpy as np
import pickle

In [2]:
# fasttext.util.download_model('en', if_exists='ignore') 

In [3]:

ft_model = fasttext.load_model('model/cc.en.300.bin')

In [11]:
input_file = "dataset/sentences.txt"
output_file = "embedding/fasttext_embeddings.pkl"

In [ ]:
def get_sentence_embedding(sentence: str) -> np.ndarray:

    words = sentence.strip().split()
    if not words:
        return np.zeros(ft_model.get_dimension())
    
    word_vectors = [ft_model.get_word_vector(word) for word in words]
    return np.mean(word_vectors, axis=0)  


In [6]:
# Read sentences
with open(input_file, "r", encoding="utf-8") as f:
    sentences = [line.strip() for line in f if line.strip()]

In [7]:
# Generate embeddings
print(f"Generating FastText embeddings for {len(sentences)} sentences...")
embeddings = np.array([get_sentence_embedding(sentence) for sentence in sentences])
print(f"Embeddings shape: {embeddings.shape}")  # (num_sentences, 300)


Generating FastText embeddings for 3408 sentences...
Embeddings shape: (3408, 300)


In [12]:
# Save embeddings (same format as before)
with open(output_file, "wb") as f:
    pickle.dump(embeddings, f)

print(f"Embeddings saved to {output_file}")

Embeddings saved to embedding/fasttext_embeddings.pkl


## 3. Using Word2Vec  

In [1]:
import torch
import numpy as np
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from tools.word2vec_model import Word2VecEmbedder
from tools.word2vec_model import preprocess_financial

In [10]:
input_file  = "dataset/sentences.txt"
output_file = "embedding/w2v_embeddings.pkl"
MODEL_PATH  = "model/word2vec.pt"


In [11]:


checkpoint  = torch.load(MODEL_PATH, map_location=torch.device("cpu"))
weights     = checkpoint["weights"]
word_to_idx = checkpoint["word_to_idx"]
DIM         = checkpoint["dim"]

model = Word2VecEmbedder(weights, freeze=True)
model.eval()


Word2VecEmbedder(
  (embedding): Embedding(3000000, 300, padding_idx=0)
)

In [12]:
with open(input_file, "r", encoding="utf-8") as f:
    raw_sentences = [line.strip() for line in f if line.strip()]

sentences = [preprocess_financial(s) for s in raw_sentences]
print(f"Total sentences: {len(sentences)}")


Total sentences: 3408


In [13]:

tfidf = TfidfVectorizer(
    max_features = 50000,
    sublinear_tf = True,
    min_df       = 2,
    stop_words   = None
)
tfidf.fit(sentences)
vocab_tfidf = tfidf.vocabulary_
idf_weights = tfidf.idf_


In [14]:

def tokenize(sentence: str, max_len: int = 64) -> tuple[list[int], list[float]]:
    words     = sentence.strip().split()[:max_len]
    token_ids = []
    tfidf_wts = []

    for word in words:
        idx = (
            word_to_idx.get(word) or
            word_to_idx.get(word.title()) or
            word_to_idx.get(word.upper()) or
            0
        )
        token_ids.append(idx)
        wt = idf_weights[vocab_tfidf[word]] if word in vocab_tfidf else 1.0
        tfidf_wts.append(wt if idx != 0 else 0.1)

    return token_ids, tfidf_wts


In [15]:

def pad_batch(batch_ids, batch_wts):
    max_len = max(len(x) for x in batch_ids)
    ids_pad = torch.zeros(len(batch_ids), max_len, dtype=torch.long)
    wts_pad = torch.zeros(len(batch_wts), max_len, dtype=torch.float32)

    for i, (ids, wts) in enumerate(zip(batch_ids, batch_wts)):
        ids_pad[i, :len(ids)] = torch.tensor(ids)
        wts_pad[i, :len(wts)] = torch.tensor(wts)

    return ids_pad, wts_pad


In [16]:

BATCH_SIZE     = 128                  
all_embeddings = []
oov_total, word_total = 0, 0

print("Generating embeddings...")
with torch.no_grad():
    for i in range(0, len(sentences), BATCH_SIZE):
        batch_sents      = sentences[i : i + BATCH_SIZE]
        batch_ids, batch_wts = [], []

        for sent in batch_sents:
            ids, wts    = tokenize(sent)
            word_total += len(ids)
            oov_total  += ids.count(0)
            batch_ids.append(ids)
            batch_wts.append(wts)

        ids_tensor, wts_tensor = pad_batch(batch_ids, batch_wts)

        embeddings = model(ids_tensor, wts_tensor)    # (batch, 300)
        all_embeddings.append(embeddings.numpy())

        if (i // BATCH_SIZE) % 10 == 0:
            print(f"  Processed {min(i + BATCH_SIZE, len(sentences))}/{len(sentences)} sentences")


Generating embeddings...
  Processed 128/3408 sentences
  Processed 1408/3408 sentences
  Processed 2688/3408 sentences


In [18]:

all_embeddings = np.vstack(all_embeddings)
oov_rate       = oov_total / max(word_total, 1) * 100

print(f"\nEmbeddings shape : {all_embeddings.shape}")
print(f"OOV rate         : {oov_rate:.2f}%")


with open(output_file, "wb") as f:
    pickle.dump(all_embeddings, f)

print(f"Saved to {output_file}")


Embeddings shape : (3408, 300)
OOV rate         : 7.18%
Saved to embedding/w2v_embeddings.pkl


In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import pickle
from tqdm import tqdm


/home/prashanth/nlp/financial_assistant/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
model = AutoModel.from_pretrained('ProsusAI/finbert')
model.eval()


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 49271.93it/s]
BertModel LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
classifier.weight            | UNEXPECTED |  | 
classifier.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [3]:

input_file = "dataset/sentences.txt"
output_file = "embedding/finbert_embeddings.pkl"
BATCH_SIZE = 16  # Keep small for CPU


In [4]:

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state  # (batch, seq_len, 768)
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * input_mask_expanded).sum(1) / input_mask_expanded.sum(1).clamp(min=1e-9)


In [5]:

def get_embeddings_batch(sentences):
    inputs = tokenizer(
        sentences,
        return_tensors='pt',
        truncation=True,
        max_length=512,
        padding=True
    )
    with torch.no_grad():
        outputs = model(**inputs)
    embeddings = mean_pooling(outputs, inputs['attention_mask'])
    return embeddings.numpy()


In [6]:

# Read sentences
with open(input_file, 'r') as f:
    sentences = [line.strip() for line in f if line.strip()]

print(f"Total sentences: {len(sentences)}")


Total sentences: 3408


In [ ]:
# Process in batches
all_embeddings = []
for i in tqdm(range(0, len(sentences), BATCH_SIZE), desc="Generating embeddings"):
    batch = sentences[i : i + BATCH_SIZE]
    batch_embeddings = get_embeddings_batch(batch)
    all_embeddings.extend(batch_embeddings)


Generating embeddings: 100%|██████████| 213/213 [01:39<00:00,  2.15it/s]


In [ ]:
# Save
with open(output_file, 'wb') as f:
    pickle.dump(all_embeddings, f)

print(f"Saved {len(all_embeddings)} embeddings → {output_file}")

Saved 3408 embeddings → embedding/finbert_embeddings.pkl
